# **YOLOv11 & RT-DETR for Railroad Tie/Sleeper Defect Detection**

---
- Jupyter notebooks with EDA, training (with cross validation) and testing pipelines

## **Importing Dataset from Google Drive**

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount =True)

Mounted at /content/drive


In [ ]:
os.listdir("/content/drive")

['MyDrive', '.shortcut-targets-by-id', '.Trash-0', '.Encrypted']

In [ ]:
os.chdir("/content/drive/MyDrive")
print(f"Current CWD: {os.getcwd()}")

os.listdir(os.getcwd())
os.chdir('nsf-reu-advaychandramouli')
print(f"Dataset folder: {os.getcwd()}")

Current CWD: /content/drive/MyDrive
Dataset folder: /content/drive/MyDrive/nsf-reu-advaychandramouli


### **Verifying Subdirectories of Image & Annotations**

In [ ]:
os.listdir(os.getcwd())

['Dataset', 'Railroads_TieDefectDetection.ipynb']

In [ ]:
DATA_ROOT = os.path.join(os.getcwd(), 'Dataset')
print(f"Data root folder: {DATA_ROOT}")

os.listdir(DATA_ROOT)

Data root folder: /content/drive/MyDrive/nsf-reu-advaychandramouli/Dataset


['Railroads-YOLO11-TXT', 'folds.json', 'Deployment-DemoFootage-1min.mp4']

In [ ]:
YOLO_DATA_ROOT = os.path.join(DATA_ROOT, "Railroads-YOLO11-TXT")
print(f"YOLO Dataset Path: {YOLO_DATA_ROOT}")

YOLO Dataset Path: /content/drive/MyDrive/nsf-reu-advaychandramouli/Dataset/Railroads-YOLO11-TXT


### **Assessing Class/Label Distributions**

In [ ]:
#Iterating through YOLO Label (TXT files)
concrete_count, wood_count = 0, 0

# Concrete Sleeper/Tie ID = 1, Wood ID = 6
LABEL_ROOT = os.path.join(YOLO_DATA_ROOT, "labels")
label_files = os.listdir(LABEL_ROOT)

#If label_files
print(f"Number of labels: {len(label_files)}")

Number of labels: 1004


In [ ]:
'''
For each file in label_files,
Check if the leftmost number in the file contains either a 1 or 6
If so, increment the respective counter
'''

for label in label_files:
    # Create the path
    label_path = os.path.join(LABEL_ROOT, label)

    # Skip if path is a directory
    if os.path.isdir(label_path):
        continue

    # Open/Read the File
    classes = []
    try:
        with open(label_path, 'r', encoding='latin-1') as f: # Added encoding='latin-1'
            for line in f:
                class_id = int(line.strip().split()[0])
                classes.append(class_id)
    except Exception as e:
        print(f"Error reading file {label}: {e}")
        continue

    # Check if it contains 1 or 6
    if 1 in classes:
        concrete_count += 1
    elif 6 in classes:
        wood_count += 1
    elif len(classes) == 0:
        print(f"File with no annotations: {label}")

File with no annotations: crosstie_frame00031_jpg.rf.decfd31706504cde4e26152dc04938b6.txt
File with no annotations: crosstie_frame00136_jpg.rf.418a4911031699408ead7101e3ea626f.txt
File with no annotations: crosstie_frame00136_jpg.rf.960a15cfd0ba0b1d01d5b90528f32113.txt
File with no annotations: crosstie_frame00106_jpg.rf.71c2994169dffef609680b54732a3268.txt
File with no annotations: crosstie_frame00121_jpg.rf.da4c8a87d4f83c75cc7439e9ceb39d30.txt
File with no annotations: crosstie_frame00106_jpg.rf.7b616f3969b6e32188cc11a4017f0af6.txt
File with no annotations: crosstie_frame00076_jpg.rf.cad5d34c335d43d0059375847c5377f1.txt
File with no annotations: crosstie_frame00121_jpg.rf.dbd4c323c74a75ea74853a889f65c3b9.txt
File with no annotations: crosstie_frame00031_jpg.rf.18fb581856cb3d9977011624b28495ca.txt
File with no annotations: crosstie_frame00586_jpg.rf.960392754d9cbf98c262fe13abf44231.txt
File with no annotations: crosstie_frame00076_jpg.rf.6cc18a3ba3c4ac1eaf739b00173aafd7.txt
File with 

## **K-Fold Cross Validation Splits**

- Using K=5 Folds; This means we're training both models 5 times and averaging their performance across folds in our statistical evaluations.
- 80-20 Train Test Split
- The following cells define a few functions to dynamically shuffle image samples and labels inside our folds, to be called at training

In [ ]:
import random
from sklearn.model_selection import KFold
yolo_img_dir = os.path.join(YOLO_DATA_ROOT, "images")
yolo_img_paths = os.listdir(yolo_img_dir)

# If directories present in list remove
yolo_img_paths = [path for path in yolo_img_paths if os.path.isfile(os.path.join(yolo_img_dir, path))]

print(f"Verifying # of YOLO paths is: {len(yolo_img_paths)}")

Verifying # of YOLO paths is: 1000


In [ ]:
kf_indices = list(range(len(yolo_img_paths)))

# Seed the generator
random.seed(0)

#Create the KFold object
kf = KFold(n_splits = 5, shuffle = True, random_state = 4)
kf.split(kf_indices)

folds = {}

for fold_num, (train_idx, val_idx) in enumerate(kf.split(kf_indices)):
    # train_idx and val_idx are NP Arrays
    print(f"Dimensions of train_idx: {train_idx.ndim} | Dimensions of val-idx: {val_idx.ndim}")

    training_img_paths, val_img_paths = [], []
    training_label_paths, val_label_paths = [], []

    for index in train_idx:
        image_path = yolo_img_paths[index]
        label_path = image_path.replace('.jpg', '.txt')

        training_img_paths.append(image_path)
        training_label_paths.append(label_path)

    for index in val_idx:
        image_path = yolo_img_paths[index]
        label_path = image_path.replace('.jpg', '.txt')

        val_img_paths.append(image_path)
        val_label_paths.append(label_path)

    print(f"Fold Number: {fold_num}")
    print(f"Number of elements in training data: {len(training_img_paths)} --- {len(training_label_paths)}")
    print(f"Number of elements in validation data: {len(val_img_paths)} --- {len(val_img_paths)}")
    print()

    folds[fold_num] = {
        'train_img' : training_img_paths,
        'train_label' : training_label_paths,
        'val_img' : val_img_paths,
        'val_label' : val_label_paths
    }

Dimensions of train_idx: 1 | Dimensions of val-idx: 1
Fold Number: 0
Number of elements in training data: 800 --- 800
Number of elements in validation data: 200 --- 200

Dimensions of train_idx: 1 | Dimensions of val-idx: 1
Fold Number: 1
Number of elements in training data: 800 --- 800
Number of elements in validation data: 200 --- 200

Dimensions of train_idx: 1 | Dimensions of val-idx: 1
Fold Number: 2
Number of elements in training data: 800 --- 800
Number of elements in validation data: 200 --- 200

Dimensions of train_idx: 1 | Dimensions of val-idx: 1
Fold Number: 3
Number of elements in training data: 800 --- 800
Number of elements in validation data: 200 --- 200

Dimensions of train_idx: 1 | Dimensions of val-idx: 1
Fold Number: 4
Number of elements in training data: 800 --- 800
Number of elements in validation data: 200 --- 200



In [ ]:
print(f"Created {len(folds)} folds")
for fold_key in folds.keys():
    print(f"{fold_key}: {len(folds[fold_key]['train_img'])} train, {len(folds[fold_key]['val_img'])} val")

Created 5 folds
0: 800 train, 200 val
1: 800 train, 200 val
2: 800 train, 200 val
3: 800 train, 200 val
4: 800 train, 200 val


In [ ]:
os.chdir(YOLO_DATA_ROOT)
os.getcwd()

'/content/drive/MyDrive/nsf-reu-advaychandramouli/Dataset/Railroads-YOLO11-TXT'

- Creating subdirectories within images and labels folders for:
    - Training samples: expect 800 images & labels
    - Testing samples: expect 200 images & labels

In [ ]:
# Create Train & Val folders within Images and Labels

subdir_target = ['images', 'labels']
subdir_structure = ['train', 'val']
for target in subdir_target:
    cur_path = os.path.join(YOLO_DATA_ROOT, target)

    for folder in subdir_structure:
        target_path = os.path.join(cur_path, folder)
        print(f"Created {folder} folder inside {target}...")
        if not os.path.exists(target_path):
            os.mkdir(target_path)

Created train folder inside images...
Created val folder inside images...
Created train folder inside labels...
Created val folder inside labels...


In [ ]:
def clear_tempfolders():
    IMG_ROOT = os.path.join(YOLO_DATA_ROOT, 'images')
    LABEL_ROOT = os.path.join(YOLO_DATA_ROOT, 'labels')

    root_dirs = [IMG_ROOT, LABEL_ROOT]
    for root in root_dirs:
        targets = ["train", "val"]
        for target in targets:
            curr_dir = os.path.join(root, target)
            for filename in os.listdir(curr_dir):
                os.remove(os.path.join(curr_dir, filename))
            print(f"Successfully emptied folders -- Root: {root} | Target: {target} | Current length: {len(os.listdir(curr_dir))} ")

In [ ]:
import shutil

def create_folds(fold_index):
    clear_tempfolders()

    IMG_ROOT = os.path.join(YOLO_DATA_ROOT, 'images')
    LABEL_ROOT = os.path.join(YOLO_DATA_ROOT, 'labels')
    path_list = []

    for key in folds[fold_index]:
        dir_params = key.split('_')
        if 'img' in dir_params:
            if dir_params[0] == 'train':
                path_list = folds[fold_index]['train_img']
            else:
                path_list = folds[fold_index]['val_img']

            for path in path_list:
                src = os.path.join(IMG_ROOT, path)
                dst = os.path.join(IMG_ROOT, dir_params[0], path)

                shutil.copy(src, dst)
        else:
            if dir_params[0] == 'train':
                path_list = folds[fold_index]['train_label']
            else:
                path_list = folds[fold_index]['val_label']

            for path in path_list:
                src = os.path.join(LABEL_ROOT, path)
                dst = os.path.join(LABEL_ROOT, dir_params[0], path)

                shutil.copy(src, dst)

In [ ]:
create_folds(0)

Successfully emptied folders -- Root: /content/drive/MyDrive/nsf-reu-advaychandramouli/Dataset/Railroads-YOLO11-TXT/images | Target: train | Current length: 0 
Successfully emptied folders -- Root: /content/drive/MyDrive/nsf-reu-advaychandramouli/Dataset/Railroads-YOLO11-TXT/images | Target: val | Current length: 0 
Successfully emptied folders -- Root: /content/drive/MyDrive/nsf-reu-advaychandramouli/Dataset/Railroads-YOLO11-TXT/labels | Target: train | Current length: 0 
Successfully emptied folders -- Root: /content/drive/MyDrive/nsf-reu-advaychandramouli/Dataset/Railroads-YOLO11-TXT/labels | Target: val | Current length: 0 


##**Model Configuration**
- These cells create a YAML file documenting information about our dataset such as:
    - Classes & their ordinal encoding (0, 1, 2 for each class respectively)
    - Paths for training and validation samples (both labels & images)
    - Paths for dataset root folder

In [ ]:
!pip install pyyaml

In [ ]:
import os, yaml

def create_yaml(data_root):
    data = {
        'path': data_root,
        'train': "images/train",
        'val': "images/val",
        'nc': 3,  # Reduced from 7 to 3 classes
        'names': {
            0: 'Wood Check',
            1: 'Wood Decay',
            2: 'Wood Sleeper'
        },
        'roboflow': {
            'workspace': '2025-nsf-reu-railroad-defect-detection',
            'project': 'nsf-reu-urda',
            'version': 4,
            'license': 'MIT',
            'url': 'https://universe.roboflow.com/2025-nsf-reu-railroad-defect-detection/nsf-reu-urda/dataset/4'
        }
    }

    dst = os.path.join(data_root, "config.yaml")
    try:
        with open(dst, 'w') as f:
            yaml.dump(data, f, default_flow_style=False)
        print(f"YAML file created at: {dst}")
    except Exception as e:
        print(f"Error creating YAML file: {e}")

create_yaml("/content/drive/MyDrive/nsf-reu-advaychandramouli/Dataset/Railroads-YOLO11-TXT")


YAML file created at: /content/drive/MyDrive/nsf-reu-advaychandramouli/Dataset/Railroads-YOLO11-TXT/config.yaml


## **Model Training/Testing**
- These cells are where we define our training loop, where:
    - Before each training run, we clear the folders containing our images and dataset to ensure a fair/representative shuffle
    - Call the requisite functions for partitioning our dataset for that fold
    - Set fixed hyperparameters (epochs, batch sizes, optimizer)
    - Train both YOLOv11-L and RT-DETR consecutively, on the same folds of data for fairness
    - Store each model's performance metrics in a hashmap where the key corresponds to the fold #, and the values are the series of statistics for that iteration. This will come in useful for post-training model evaluation.



In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO, RTDETR
import os

# Models to test
large_models_config = {
    'yolo11': 'yolo11l.pt',
    'rtdetr': 'rtdetr-l.pt',
}

# Hyperparameters
epoch_range = [500]
batch_sizes = [32]

k_folds = 5
yolo_results = []
detr_results = []

total_experiments = k_folds * len(epoch_range) * len(batch_sizes)
experiment_count = 0

for fold_idx in range(k_folds):
    create_folds(fold_idx)
    print(f"Successfully Created Fold: {fold_idx + 1}")

    for ep in epoch_range:
        for batch in batch_sizes:
            experiment_count += 1
            print("")
            print(f"--- Experiment {experiment_count}/{total_experiments} ---")
            print(f"Fold: {fold_idx + 1}, Epochs: {ep}, Batch: {batch}")

            # Test YOLO11-Large
            print("Training YOLO11-Large...")
            yolo_model = YOLO(large_models_config['yolo11'])

            yolo_fold_results = yolo_model.train(
                data='/content/drive/MyDrive/nsf-reu-advaychandramouli/Dataset/Railroads-YOLO11-TXT/config.yaml',
                epochs=ep,
                batch=batch,    # Fixed: now using the parameter
                lr0=0.02,
                optimizer='SGD',
                plots=True,
                val=True
            )

            # Store YOLO results with hyperparameter info
            yolo_metrics = yolo_fold_results if isinstance(yolo_fold_results, dict) else yolo_fold_results.results_dict
            yolo_metrics['fold'] = fold_idx
            yolo_metrics['epochs'] = ep
            yolo_metrics['batch_size'] = batch
            yolo_results.append(yolo_metrics)

            # Test RT-DETR-Large
            print("Training RT-DETR-Large...")
            detr_model = RTDETR(large_models_config['rtdetr'])

            detr_fold_results = detr_model.train(
                data='/content/drive/MyDrive/nsf-reu-advaychandramouli/Dataset/Railroads-YOLO11-TXT/config.yaml',
                epochs=ep,      # Fixed: now using the parameter
                batch=batch,    # Fixed: now using the parameter
                lr0=0.02,
                optimizer='SGD',
                plots=True,
                val=True
            )

            # Store RT-DETR results with hyperparameter info
            detr_metrics = detr_fold_results if isinstance(detr_fold_results, dict) else detr_fold_results.results_dict
            detr_metrics['fold'] = fold_idx
            detr_metrics['epochs'] = ep
            detr_metrics['batch_size'] = batch
            detr_results.append(detr_metrics)

print(f"\nCompleted all {total_experiments} experiments!")
print(f"YOLO results: {len(yolo_results)} experiments")
print(f"DETR results: {len(detr_results)} experiments")

print("\nSample YOLO Result:")
print(yolo_results[0])
print("\nSample DETR Result:")
print(detr_results[0])

Output hidden; open in https://colab.research.google.com to view.

## **Model Evaluation**

- Since we've stored our model results to 2 different hashmaps, with the same structure, the following cells:
    - Define functions to average each performance metric across 5 folds
    - Compare the average performance across metrics for each model and determine which model performed better for each metric
    - Print these results to console in a tabular format for analysis/interpretation purposes


In [ ]:
import numpy as np

def calculate_average_metrics(results_list, model_name):
    """
    Calculate average metrics across all folds for a given model

    Args:
        results_list: List of dictionaries containing metrics for each fold
        model_name: String name of the model for display

    Returns:
        Dictionary of averaged metrics
    """
    if not results_list:
        print(f"No results found for {model_name}")
        return {}

    # Get all metric keys from the first fold
    metric_keys = results_list[0].keys()
    averaged_metrics = {}

    # Calculate average for each metric across all folds
    for metric in metric_keys:
        values = []
        for fold_result in results_list:
            if metric in fold_result and fold_result[metric] is not None:
                values.append(fold_result[metric])

        if values:
            averaged_metrics[metric] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'min': np.min(values),
                'max': np.max(values)
            }

    return averaged_metrics

def print_model_performance(averaged_metrics, model_name):
    """
    Pretty print the averaged model performance
    """
    print(f"\n{'='*50}")
    print(f"{model_name.upper()} - AVERAGE PERFORMANCE")
    print(f"{'='*50}")

    # Key metrics to highlight (common YOLO/DETR metrics)
    key_metrics = ['metrics/mAP50-95(B)', 'metrics/mAP50(B)', 'metrics/precision(B)', 'metrics/recall(B)']

    # Print key metrics first
    print("KEY METRICS:")
    print("-" * 30)
    for metric in key_metrics:
        if metric in averaged_metrics:
            stats = averaged_metrics[metric]
            print(f"{metric:25}: {stats['mean']:.4f} ± {stats['std']:.4f} (min: {stats['min']:.4f}, max: {stats['max']:.4f})")

    # Print all other metrics
    print("\nALL METRICS:")
    print("-" * 30)
    for metric, stats in averaged_metrics.items():
        if metric not in key_metrics:
            print(f"{metric:25}: {stats['mean']:.4f} ± {stats['std']:.4f}")

def compare_models(yolo_metrics, detr_metrics):
    """
    Compare key metrics between YOLO and DETR models
    """
    print(f"\n{'='*60}")
    print("MODEL COMPARISON")
    print(f"{'='*60}")

    key_metrics = ['metrics/mAP50-95(B)', 'metrics/mAP50(B)', 'metrics/precision(B)', 'metrics/recall(B)']

    print(f"{'Metric':<25} {'YOLO11-L':<15} {'RT-DETR-L':<15} {'Winner':<10}")
    print("-" * 70)

    for metric in key_metrics:
        if metric in yolo_metrics and metric in detr_metrics:
            yolo_val = yolo_metrics[metric]['mean']
            detr_val = detr_metrics[metric]['mean']
            winner = "YOLO" if yolo_val > detr_val else "DETR"

            print(f"{metric:<25} {yolo_val:<15.4f} {detr_val:<15.4f} {winner:<10}")

# Main evaluation script
def evaluate_models(yolo_results, detr_results):
    """
    Main function to evaluate and compare models
    """
    # Calculate average metrics for each model
    yolo_avg_metrics = calculate_average_metrics(yolo_results, "YOLO11-Large")
    detr_avg_metrics = calculate_average_metrics(detr_results, "RT-DETR-Large")

    # Print individual model performances
    print_model_performance(yolo_avg_metrics, "YOLO11-Large")
    print_model_performance(detr_avg_metrics, "RT-DETR-Large")

    # Compare models
    compare_models(yolo_avg_metrics, detr_avg_metrics)

    return yolo_avg_metrics, detr_avg_metrics

# Usage example:

In [ ]:
yolo_avg, detr_avg = evaluate_models(yolo_results, detr_results)


YOLO11-LARGE - AVERAGE PERFORMANCE
KEY METRICS:
------------------------------
metrics/mAP50-95(B)      : 0.9014 ± 0.0134 (min: 0.8819, max: 0.9183)
metrics/mAP50(B)         : 0.9530 ± 0.0106 (min: 0.9427, max: 0.9683)
metrics/precision(B)     : 0.9696 ± 0.0077 (min: 0.9596, max: 0.9785)
metrics/recall(B)        : 0.9104 ± 0.0147 (min: 0.8944, max: 0.9339)

ALL METRICS:
------------------------------
fitness                  : 0.9066 ± 0.0130
fold                     : 2.0000 ± 1.4142
epochs                   : 500.0000 ± 0.0000
batch_size               : 32.0000 ± 0.0000

RT-DETR-LARGE - AVERAGE PERFORMANCE
KEY METRICS:
------------------------------
metrics/mAP50-95(B)      : 0.7898 ± 0.0131 (min: 0.7705, max: 0.8102)
metrics/mAP50(B)         : 0.9321 ± 0.0094 (min: 0.9175, max: 0.9425)
metrics/precision(B)     : 0.9498 ± 0.0088 (min: 0.9393, max: 0.9604)
metrics/recall(B)        : 0.9119 ± 0.0152 (min: 0.8910, max: 0.9329)

ALL METRICS:
------------------------------
fitness       